# Chapter 1: Decision Trees

> **Learning means generalizing beyond training data, not memorizing it.** The key is to distinguish between training error and test error.

**Type:** Learn + Build &nbsp;|&nbsp; **Language:** Python &nbsp;|&nbsp; **Prerequisites:** None &nbsp;|&nbsp; **Time:** ~30 minutes
**Source:** *A Course in Machine Learning*, Hal Daumé III — Chapter 1

---

## Learning Objectives

- Explain the difference between memorization and **generalization**
- Define **inductive bias** and recognize its role in learning
- Cast a concrete task as a formal learning problem (input space, features, output space, loss function)
- Illustrate how **regularization** (via `max_depth`) trades off underfitting vs. overfitting
- Evaluate whether using test data is "cheating" or not

## The Problem

**Scenario:** You're building a course recommendation system. Given a student's past course ratings and characteristics (easy course? AI-related? morning time slot?), predict whether they will like a new course.

**Challenge:** How do you learn a prediction rule from past examples that will generalize to unseen students?

**Key Insight:** If you simply memorize past examples, your learned function will fail on new data. You need to find patterns that *generalize*.

## The Concept

### Core Idea: Divide and Conquer

A **decision tree** repeatedly asks binary questions about features, partitioning the data until each region is "pure" (contains mostly one label).

```
                   [Is course in Systems?]
                    /                    \
                  NO                     YES
                /                          \
         [Easy?]                     [Hard-margin SVM?]
         /      \                         /
       NO       YES                      ...
      ...       ...
```

**Algorithm 1 — DecisionTreeTrain** (Chapter 1, Algorithm 1.3)
1. Start with all examples and all features available
2. **Base case:** if labels are pure → return a leaf with the majority label
3. **Recursive case:** find the feature that best splits the data (highest accuracy if we split here)
4. Partition the data by that feature; remove it from future use
5. Recursively build the left subtree (feature = 0) and the right subtree (feature = 1)

**Algorithm 2 — DecisionTreeTest** (Chapter 1, Algorithm 1.3)
1. Start at the root
2. If it's a leaf → return its guess
3. If it's a node → check the feature value in the test point
4. Go left (feature = 0) or right (feature = 1) and recurse

### Key Concepts

| Concept | Meaning |
|---|---|
| **Feature** | A question you can ask (e.g., "Is this a Systems course?") |
| **Label** | The correct answer (e.g., +1 for "liked", -1 for "disliked") |
| **Training error** | Accuracy on data the model saw |
| **Test error** | Accuracy on new, unseen data ← *this is what matters* |
| **Generalization** | Model does well on test data, not just train data |
| **Inductive bias** | The model assumes certain patterns are "simple" and likely |

## Build It

### Setup

We'll need NumPy for array operations, `Counter` for tallying labels, and two utilities from scikit-learn: the Breast Cancer Wisconsin dataset and a train/test splitter. We'll also import scikit-learn's own `DecisionTreeClassifier` later, purely as a sanity check against our from-scratch implementation.

In [1]:
import numpy as np
from collections import Counter
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

### Step 1: Represent the Tree

A decision tree is made of two kinds of nodes:

- **`Leaf`** — holds a fixed prediction (`guess`)
- **`Node`** — asks a binary question on one `feature`, then routes to a `left` subtree (answer = 0 / "no") or a `right` subtree (answer = 1 / "yes")

In [2]:
class Leaf:
    def __init__(self, guess):
        self.guess = guess

    def __repr__(self):
        return f"Leaf({self.guess})"


class Node:
    def __init__(self, feature, left, right):
        self.feature = feature
        self.left = left
        self.right = right

    def __repr__(self):
        return f"Node(f={self.feature})"


### Step 2: A Small Helper — Majority Vote

Every leaf needs a prediction. `most_frequent_label` simply returns whichever label appears most often in a given set — this is the guess a leaf falls back on.

In [3]:
def most_frequent_label(y):
    if len(y) == 0:
        return None
    counts = Counter(y)
    return counts.most_common(1)[0][0]

### Step 3: Find the Best Feature (Greedy Split)

At each step, the algorithm tries every remaining feature and asks: *if I split the data on this feature, how many examples would I classify correctly (using the majority label on each side)?*

$$\text{score} = \#\{\text{correct in "no" group}\} + \#\{\text{correct in "yes" group}\}$$

The feature with the highest score is chosen as the split.

In [4]:
def find_best_split(X, y, remaining_features):
    best_feature = None
    best_score = -1

    for feature_idx in remaining_features:
        no_mask = (X[:, feature_idx] == 0)
        yes_mask = (X[:, feature_idx] == 1)

        no_y = y[no_mask]
        yes_y = y[yes_mask]

        if len(no_y) == 0 or len(yes_y) == 0:
            continue

        no_guess = most_frequent_label(no_y)
        yes_guess = most_frequent_label(yes_y)

        score = np.sum(no_y == no_guess) + np.sum(yes_y == yes_guess)

        if score > best_score:
            best_score = score
            best_feature = feature_idx

    return best_feature

### Step 4: Algorithm 1 — `DecisionTreeTrain`

This is the recursive, greedy divide-and-conquer routine described above. Three base cases stop the recursion:

1. The labels are already pure (all the same)
2. There are no features left to split on
3. The tree has reached `max_depth` — this is the **regularization** hyperparameter from Section 1.9, and it's the main tool we'll use to control underfitting vs. overfitting.

In [5]:
def decision_tree_train(X, y, remaining_features, depth=0, max_depth=None):
    guess = most_frequent_label(y)

    if len(np.unique(y)) <= 1:
        return Leaf(guess)

    if len(remaining_features) == 0:
        return Leaf(guess)

    if max_depth is not None and depth >= max_depth:
        return Leaf(guess)

    best_feature = find_best_split(X, y, remaining_features)

    if best_feature is None:
        return Leaf(guess)

    no_mask = (X[:, best_feature] == 0)
    yes_mask = (X[:, best_feature] == 1)

    remaining_next = [f for f in remaining_features if f != best_feature]

    left = decision_tree_train(X[no_mask], y[no_mask], remaining_next,
                               depth + 1, max_depth)
    right = decision_tree_train(X[yes_mask], y[yes_mask], remaining_next,
                                depth + 1, max_depth)

    return Node(best_feature, left, right)

### Step 5: Algorithm 2 — `DecisionTreeTest`

Prediction is just a walk down the tree: at each `Node`, check the test point's value for that node's feature, and recurse left or right accordingly. When a `Leaf` is reached, return its stored guess.

In [6]:
def decision_tree_predict_single(tree, x):
    if isinstance(tree, Leaf):
        return tree.guess

    if x[tree.feature] == 1:
        return decision_tree_predict_single(tree.right, x)
    else:
        return decision_tree_predict_single(tree.left, x)


def decision_tree_predict(tree, X):
    return np.array([decision_tree_predict_single(tree, x) for x in X])


def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

## Use It — Real Data

### Loading and Preparing the Dataset

We'll use the **Breast Cancer Wisconsin (Diagnostic)** dataset, a real, well-known binary classification benchmark bundled with scikit-learn.

Our algorithm expects **binary features** (0 or 1), but this dataset's features are continuous measurements (radius, texture, area, etc.). We binarize each feature at its **median**: values above the median become 1, values at or below become 0. Labels are also remapped to the book's convention: −1 / +1 instead of 0 / 1.

In [7]:
data = load_breast_cancer()
X_raw = data.data
y_raw = data.target
feature_names = data.feature_names

medians = np.median(X_raw, axis=0)
X_binary = (X_raw > medians).astype(int)

y = np.where(y_raw == 0, -1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X_binary, y, test_size=0.2, random_state=42, stratify=y
)

print("DATASET: Breast Cancer Wisconsin (Diagnostic)")
print(f"  Total examples: {X_binary.shape[0]}")
print(f"  Features (binarized): {X_binary.shape[1]}")
print(f"  Train/test split: {X_train.shape[0]} / {X_test.shape[0]}")
print(f"  Class distribution: {np.sum(y == -1)} negative, {np.sum(y == 1)} positive")

DATASET: Breast Cancer Wisconsin (Diagnostic)
  Total examples: 569
  Features (binarized): 30
  Train/test split: 455 / 114
  Class distribution: 212 negative, 357 positive


### Section 1.7 & 1.9: The Underfitting / Overfitting Tradeoff

Now for the central experiment of the chapter. We train trees with increasing `max_depth` values and compare **training accuracy** to **test accuracy**:

- A **shallow** tree (small `max_depth`) may be too simple to capture real patterns → **underfitting**
- A **very deep** tree (or `max_depth=None`, unlimited) can memorize the training set, including its noise → **overfitting**
- Somewhere in between lies a **sweet spot**, where test accuracy peaks

In [8]:
all_features = list(range(X_train.shape[1]))
results = []

print(f"{'max_depth':>10} | {'train_acc':>10} | {'test_acc':>10} | {'gap':>10}")
print("-" * 50)

for depth in [1, 2, 3, 4, 5, 7, 10, None]:
    tree = decision_tree_train(X_train, y_train, all_features, max_depth=depth)

    train_acc = accuracy(y_train, decision_tree_predict(tree, X_train))
    test_acc = accuracy(y_test, decision_tree_predict(tree, X_test))

    results.append({'depth': depth, 'train_acc': train_acc, 'test_acc': test_acc})

    gap = train_acc - test_acc
    depth_str = "None" if depth is None else str(depth)
    print(f"{depth_str:>10} | {train_acc:>10.4f} | {test_acc:>10.4f} | {gap:>10.4f}")

best_result = max(results, key=lambda r: r['test_acc'])
print()
print(f"BEST: max_depth={best_result['depth']} achieves test_acc={best_result['test_acc']:.4f}")

 max_depth |  train_acc |   test_acc |        gap
--------------------------------------------------
         1 |     0.8615 |     0.7632 |     0.0984
         2 |     0.9209 |     0.8772 |     0.0437
         3 |     0.9275 |     0.9298 |    -0.0024
         4 |     0.9429 |     0.9123 |     0.0306
         5 |     0.9538 |     0.9211 |     0.0328
         7 |     0.9560 |     0.9211 |     0.0350
        10 |     0.9868 |     0.9298 |     0.0570
      None |     0.9978 |     0.9386 |     0.0592

BEST: max_depth=None achieves test_acc=0.9386


**Reading the table:**

- **`max_depth` = 1–2:** underfitting (high bias) — the model is too simple to capture the real decision boundary
- **`max_depth` = 3–5:** typically the sweet spot — a good balance of bias and variance
- **`max_depth` = 7+ / `None`:** overfitting (high variance) — training accuracy keeps climbing while test accuracy stalls or drops; the tree is memorizing noise

### Sanity Check Against `sklearn.tree.DecisionTreeClassifier`

To validate the from-scratch implementation, we compare it against scikit-learn's production decision tree, using the best `max_depth` found above.

Note that the two are **not expected to match exactly**: our splitting rule scores candidate features by raw majority-vote accuracy, while scikit-learn's default (`criterion='gini'`) scores splits by Gini impurity / information gain. Different criteria can select different splits, especially deeper in the tree — so a close (not identical) accuracy is the expected outcome.

In [9]:
best_depth = best_result['depth']

tree_ours = decision_tree_train(X_train, y_train, all_features, max_depth=best_depth)
our_test_acc = accuracy(y_test, decision_tree_predict(tree_ours, X_test))

sk_tree = DecisionTreeClassifier(criterion='gini', max_depth=best_depth, random_state=42)
sk_tree.fit(X_train, y_train)
sk_test_acc = accuracy(y_test, sk_tree.predict(X_test))

print(f"Our implementation   (max_depth={best_depth}): test accuracy = {our_test_acc:.4f}")
print(f"sklearn's classifier (max_depth={best_depth}): test accuracy = {sk_test_acc:.4f}")
print(f"Difference: {abs(our_test_acc - sk_test_acc):.4f}")

Our implementation   (max_depth=None): test accuracy = 0.9386
sklearn's classifier (max_depth=None): test accuracy = 0.9474
Difference: 0.0088


### Section 1.3: Which Feature Does the Root Split On?

The root node's feature is whichever single question, out of all 30 available, most improves majority-vote accuracy on the very first split. Inspecting it gives an interpretable first insight into the dataset.

In [10]:
if isinstance(tree_ours, Node):
    root_feature_name = feature_names[tree_ours.feature]
    print(f"Root node splits on feature: '{root_feature_name}'")
    print("(This is the feature that maximizes accuracy on the first split)")

Root node splits on feature: 'worst concave points'
(This is the feature that maximizes accuracy on the first split)


## When to Use Decision Trees

✅ Interpretability is critical
✅ Mixed feature types (continuous, categorical)
✅ Many irrelevant features
✅ Speed is critical at test time

❌ Very large datasets (training is slow)
❌ XOR-like problems (purely non-linear patterns a single tree struggles to isolate)

## Key Terms

| Term | Meaning |
|---|---|
| **Generalization** | Test error ≈ train error |
| **Overfitting** | Train error ≪ test error (memorized noise) |
| **Underfitting** | Both train and test error high (model too simple) |
| **Regularization** | A constraint that prevents overfitting (e.g., `max_depth`) |
| **Hyperparameter** | A parameter set *before* training (e.g., `max_depth`) |

## Summary

- Decision trees learn by greedily partitioning data using binary questions
- **Generalization** (test error) is the goal, not memorization (train error)
- `max_depth` is a regularizer that controls the underfitting/overfitting tradeoff
- Our from-scratch implementation reaches a test accuracy close to scikit-learn's on the Breast Cancer Wisconsin dataset
- Trees are interpretable and fast, but can overfit if left unconstrained

---

**Next:** Chapter 2 — Geometry and Nearest Neighbors